In [ ]:
import librosa
import numpy as np
import tensorflow as tf
import joblib

# -------------------------------
# Load Optimized TFLite model
# -------------------------------
interpreter = tf.lite.Interpreter(
    model_path="cnn_model_retrain_optimized.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# -------------------------------
# Load Label Encoder
# -------------------------------
label_encoder = joblib.load("label_encoder_mp2.pkl")

# -------------------------------
# Prediction Function
# -------------------------------
def predict_alert(audio_path):
    audio_path = audio_path.strip().strip('"')

    # Load audio (MATCH TRAINING)
    audio, sr = librosa.load(audio_path, sr=22050)

    # Extract Mel Spectrogram (MATCH TRAINING)
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128
    )

    mel_spec_db = librosa.power_to_db(mel_spec)

    # Fix time dimension to 173 (MATCH TRAINING)
    mel_spec_db = librosa.util.fix_length(
        mel_spec_db,
        size=173,
        axis=1
    )

    # Add channel + batch dimension
    mel_spec_db = np.expand_dims(mel_spec_db, axis=-1)
    mel_spec_db = np.expand_dims(mel_spec_db, axis=0)
    mel_spec_db = mel_spec_db.astype(np.float32)

    # Run inference
    interpreter.set_tensor(input_details[0]['index'], mel_spec_db)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])

    print("Raw model output:", output)

    predicted_class = np.argmax(output)

    print("\nClass probabilities:")
    for i, prob in enumerate(output[0]):
        print(f"{label_encoder.classes_[i]}: {prob:.4f}")

    return label_encoder.inverse_transform([predicted_class])[0]

# -------------------------------
# Main
# -------------------------------
if __name__ == "__main__":
    audio_file = input("Enter audio file path (.wav): ")

    try:
        result = predict_alert(audio_file)
        print("\nPredicted Alert Sound:", result)
    except Exception as e:
        print("\nError:", e)